In [67]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel, VarianceThreshold, SelectPercentile
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from dotenv import load_dotenv
import joblib
from gene_thesaurus import GeneThesaurus


In [68]:
load_dotenv()

X_train_path = os.getenv("X_TRAIN_CSV_PATH")
y_train_path = os.getenv("Y_TRAIN_TSV_PATH")
X_test_path = os.getenv("X_TEST_CSV_PATH")
y_test_path = os.getenv("Y_TEST_TSV_PATH")
model_path = os.getenv("RF_MODEL_PATH")
selected_features_path = os.getenv("SELECTED_FEATURES_PATH")
hyperparameters_path = os.getenv("HYPERPARAMETERS_PATH")
pipeline_path = os.getenv("PIPELINE_PATH")
random_seed = int(os.getenv("RANDOM_SEED"))
n_jobs = int(os.getenv("N_JOBS", -1))

# Print configuration
print(f"X_train_path: {X_train_path}")
print(f"y_train_path: {y_train_path}")
print(f"X_test_path: {X_test_path}")
print(f"y_test_path: {y_test_path}")
print(f"model_path: {model_path}")
print(f"selected_features_path: {selected_features_path}")
print(f"hyperparameters_path: {hyperparameters_path}")
print(f"pipeline_path: {pipeline_path}")
print(f"random_seed: {random_seed}")
print(f"n_jobs: {n_jobs}")

# Hardcoded variables
VARIANCE_THRESHOLD=0.01  # REMOVE NEAR-CONSTANT GENES
FEATURE_SELECTION_PERCENTILE=20  # KEEP TOP n% OF FEATURES
FINAL_N_ESTIMATORS=500

X_train_path: /home/mariya/Data/pallas/nf-core-gexml-output-train/transpose/transposed.csv
y_train_path: /home/mariya/Data/pallas/nf-core-gexml-output-train/class/subtypes.tsv
X_test_path: /home/mariya/Data/pallas/nf-core-gexml-output-cal/transpose/transposed.csv
y_test_path: /home/mariya/Data/pallas/nf-core-gexml-output-cal/class/subtypes.tsv
model_path: /home/mariya/Data/pallas/training/gex/gex.model.pkl
selected_features_path: /home/mariya/Data/pallas/training/gex/gex.selected_features.pkl
hyperparameters_path: /home/mariya/Data/pallas/training/gex/gex.hyperparameters.pkl
pipeline_path: /home/mariya/Data/pallas/training/gex/gex.pipeline.pkl
random_seed: 42
n_jobs: 10


In [69]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)
X_train


,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
id,,,,,,,,,,,,,,,,,,,,,
SJALL015985_D1,-1.299576,-1.299576,4.438162,3.609288,4.561199,1.736254,2.177061,4.306577,4.157330,7.509475,...,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576
SJALL015988_D1,-1.095438,-1.299576,4.176104,4.040786,4.343118,4.141678,4.329687,1.425480,3.920489,6.643078,...,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576
SJALL050885_D1,1.641327,-1.299576,3.642968,3.640787,4.312626,5.636464,2.863616,2.507316,4.534967,6.530232,...,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576
SJAML001427_D2,-1.299576,-1.299576,4.133432,4.439963,4.150209,9.014259,-0.355545,5.942580,5.032409,6.071148,...,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576
SJAML001427_D3,-0.548342,-1.299576,3.663728,3.913209,3.795489,8.548844,-0.087777,5.283419,4.653769,5.628271,...,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ALL_S1,3.010809,-1.299576,2.851905,3.759334,5.984948,4.522720,-0.719626,2.469506,4.599323,7.315677,...,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576
ALL_Validation_20,-0.281439,-1.299576,3.746812,4.187836,4.275723,6.714445,0.345301,3.557390,4.172423,6.399716,...,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576
Constitutional_559rep2,-0.499787,-1.299576,3.448269,4.074399,3.530864,8.093753,0.533009,5.468907,5.358870,6.259973,...,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576


In [70]:
y_train = pd.read_csv(y_train_path, sep="\t", index_col=0)
y_train

,class
sample,
AML015-D,"AML, KMT2A-r"
AML092-D,"AML, CBFB::MYH11"
AML097-D,"AML, RUNX1::RUNX1T1"
AML054-D,"AML, NPM1alt"
AML095-D,"AML, NPM1alt"
...,...
SJHBBBM4PX3,"B-ALL, high hyperdiploid"
SJHBBBM4X52,"B-ALL, ETV6::RUNX1"
SJHBBBM5GW9,"B-ALL, high hyperdiploid"


In [71]:
# align the indices
X_train = X_train.loc[y_train.index]
X_train

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
sample,,,,,,,,,,,,,,,,,,,,,
AML015-D,-0.984928,-1.299576,4.019407,4.855302,4.219090,5.866853,-1.151164,4.521930,4.258611,5.911188,...,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576
AML092-D,-1.299576,-1.299576,4.748926,3.338619,2.994308,8.485207,-0.937649,5.045698,4.100084,5.255039,...,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576
AML097-D,-0.625089,-1.299576,3.935978,4.109232,4.702469,4.975465,0.908770,4.021937,4.185322,6.792041,...,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576
AML054-D,2.760716,-1.299576,4.774938,3.136259,4.900407,3.053098,1.168226,4.754717,7.049623,6.172038,...,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576
AML095-D,0.376648,-1.299576,4.468426,3.433175,3.009214,4.425760,5.575536,4.604610,4.337068,5.445986,...,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SJHBBBM4PX3,-1.162389,-1.299576,4.586001,4.540043,4.686230,5.155253,2.588716,1.994309,5.592501,5.315408,...,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576
SJHBBBM4X52,-0.930023,-1.299576,3.872169,4.467421,3.734624,8.774620,0.823897,5.218296,5.285291,5.750370,...,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576
SJHBBBM5GW9,-0.806166,-1.299576,3.992022,4.559117,3.424167,8.682488,2.253318,4.711497,2.664907,5.969675,...,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576,-1.299576


In [72]:
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, sep="\t", index_col=0)
y_test

,class
sample,
SJHYPER001,"B-ALL, high hyperdiploid"
SJBALL016242,"B-ALL, ETV6::RUNX1-like"
SJBALL046123,"B-ALL, PAX5 P80R"
SJBALL013909,"B-ALL, KMT2A-r"
SJALL050001,"B-ALL, KMT2A-r"
...,...
AML051-D,"AML, NPM1alt"
AML101-D,"AML, CBFB::MYH11"
AML011-D,"AML, RUNX1::RUNX1T1"


In [73]:
# align the indices
X_test = X_test.loc[y_test.index]
X_test

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
sample,,,,,,,,,,,,,,,,,,,,,
SJHYPER001,-0.720996,-1.289688,4.008388,4.522121,4.389194,5.755320,-1.189916,3.505191,3.627157,6.346504,...,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688
SJBALL016242,-0.993386,-1.289688,4.419841,3.548738,4.004305,4.435923,-0.098180,2.292811,4.272842,5.789655,...,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688
SJBALL046123,0.911110,-0.795108,4.407465,3.283126,4.198197,3.064495,4.841780,4.602403,5.745861,6.084148,...,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688
SJBALL013909,-0.285326,-1.289688,3.959274,2.819025,4.309300,2.930698,1.817536,5.397323,4.066121,6.479369,...,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688
SJALL050001,-0.289568,-1.289688,4.841943,4.682225,4.048874,5.865719,-0.489694,4.823726,4.758869,6.115985,...,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AML051-D,-1.289688,-1.289688,3.507583,4.625776,4.080218,7.025077,0.323295,3.920204,4.754834,6.945966,...,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688
AML101-D,-1.289688,-1.289688,4.401719,2.395061,2.830008,10.495196,-1.045819,4.916111,4.197283,4.516816,...,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688
AML011-D,-1.289688,-1.289688,4.186125,4.023948,4.551876,4.765453,-0.529445,4.476909,4.839004,5.799896,...,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688,-1.289688


In [74]:
# Get least frequently ocurring class count
min_class_count = y_train.value_counts().min()
min_class_count

np.int64(5)

In [75]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array
y_test = y_test.iloc[:, 0].values
y_test

array(['B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1-like',
       'B-ALL, PAX5 P80R', 'B-ALL, KMT2A-r', 'B-ALL, KMT2A-r',
       'B-ALL, DUX4-r', 'B-ALL, KMT2A-r', 'B-ALL, ETV6::RUNX1',
       'B-ALL, BCR::ABL1-like', 'B-ALL, DUX4-r', 'B-ALL, BCR::ABL1',
       'B-ALL, ETV6::RUNX1', 'B-ALL, ETV6::RUNX1', 'B-ALL, ETV6::RUNX1',
       'B-ALL, high hyperdiploid', 'B-ALL, DUX4-r',
       'B-ALL, BCR::ABL1-like', 'B-ALL, KMT2A-r', 'B-ALL, ZNF384-r',
       'B-ALL, BCR::ABL1-like', 'B-ALL, KMT2A-r',
       'B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
       'B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1-like',
       'B-ALL, high hyperdiploid', 'B-ALL, ZNF384-r', 'B-ALL, BCR::ABL1',
       'B-ALL, BCR::ABL1-like', 'B-ALL, high hyperdiploid',
       'B-ALL, BCR::ABL1-like', 'B-ALL, hypodiploid', 'B-ALL, NUTM1-r',
       'B-ALL, KMT2A-r', 'B-ALL, high hyperdiploid',
       'B-ALL, high hyperdiploid', 'B-ALL, BCR::ABL1',
       'B-ALL, high hyperdiploid', 'B-ALL, TCF3::PBX1',
   

In [76]:
N_ITER=30
N_REPEATS=1 # WE WILL VALIDATE FINAL PARAMS USING MORE REPEATS

# ----------------------
# Pipeline: Variance filter -> RF feature selection -> RF classifier
# ----------------------
pipeline = Pipeline([
    ("variance_filter", VarianceThreshold(threshold=VARIANCE_THRESHOLD)), # remove near-constant genes
    ("feature_selection", SelectPercentile(percentile=FEATURE_SELECTION_PERCENTILE)),  # keep only top 10% of features
    ("rf", RandomForestClassifier(random_seed, n_jobs=-n_jobs))
])

# ----------------------
# Hyperparameter search space
# ----------------------
param_grid = {
    'rf__n_estimators': [50],
    'rf__max_depth': [None, 20, 50],
    'rf__max_features': ['sqrt', 'log2'],
    'rf__min_samples_split': [5, 10, 20],
    'rf__min_samples_leaf': [2, 5, 10],
    'rf__class_weight': [None, 'balanced']
}

# ----------------------
# Cross-validation + randomized search
# ----------------------
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=N_ITER,
    cv=cv,
    scoring="f1_weighted",   # good for multiclass imbalance
    n_jobs=-n_jobs,
    random_state=random_seed,
    verbose=1
)

search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print("Best CV (F1-weighted):", search.best_score_)
best_pipeline = search.best_estimator_

## SAVE EVERYTHING
# Save best hyperparameters
joblib.dump(search.best_params_, hyperparameters_path)

# Save features
# First get features that passed the variance threshold
variance_mask = best_pipeline.named_steps['variance_filter'].get_support()
remaining_features = X_train.columns[variance_mask]

# Then get features selected by SelectFromModel
selected_mask = best_pipeline.named_steps['feature_selection'].get_support()
selected_features = remaining_features[selected_mask]

joblib.dump(selected_features, selected_features_path)

# Save the whole pipeline
joblib.dump(best_pipeline, pipeline_path)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best hyperparameters: {'rf__n_estimators': 50, 'rf__min_samples_split': 20, 'rf__min_samples_leaf': 5, 'rf__max_features': 'sqrt', 'rf__max_depth': 50, 'rf__class_weight': 'balanced'}
Best CV (F1-weighted): 0.9084380338494709


['/home/mariya/Data/pallas/training/gex/gex.pipeline.pkl']

In [77]:
# Cross-validate again with more repeats and larger estimators
N_REPEATS=10

# Use just the selected features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# Read in "best_hyperparameters.pkl"
with open(hyperparameters_path, "rb") as f:
    best_params = joblib.load(f)

best_params = {k.replace('rf__', ''): v for k, v in best_params.items()}
best_params['n_estimators'] = FINAL_N_ESTIMATORS
print("Final model parameters:", best_params)
final_rf = RandomForestClassifier(**best_params,
                            n_jobs=n_jobs,
                            random_state=random_seed)

# Cross-validate again with more repeats
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

cv_scores = cross_val_score(final_rf,
                           X_train_selected,
                           y_train,
                           cv=cv,
                           scoring="f1_weighted",
                           n_jobs=-n_jobs,
                           verbose=1)

print(f"Cross-validation F1-weighted: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Train the final model on all training data
final_rf.fit(X_train_selected, y_train)

# Save the final model
joblib.dump(final_rf, model_path)

Final model parameters: {'n_estimators': 500, 'min_samples_split': 20, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 50, 'class_weight': 'balanced'}


[Parallel(n_jobs=-10)]: Using backend LokyBackend with 5 concurrent workers.
[Parallel(n_jobs=-10)]: Done  40 tasks      | elapsed:  8.4min
[Parallel(n_jobs=-10)]: Done  50 out of  50 | elapsed: 10.4min finished


Cross-validation F1-weighted: 0.916 ± 0.011


['/home/mariya/Data/pallas/training/gex/gex.model.pkl']

In [78]:
# ----------------------
# Evaluate final model on test set
# ----------------------

# Use selected features for test set
y_pred_final = final_rf.predict(X_test_selected)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_test_selected.shape}")

print("\nTest set classification report (final optimized model):")
print(classification_report(y_test, y_pred_final, digits=3))


Number of selected features: 3924
Shape of X_test_selected: (753, 3924)

Test set classification report (final optimized model):
                          precision    recall  f1-score   support

      AML, APL-PML::RARA      1.000     1.000     1.000         1
     AML, CBFA2T3::GLIS2      1.000     1.000     1.000         7
        AML, CBFB::MYH11      0.969     1.000     0.984        31
           AML, CEBPAalt      1.000     1.000     1.000        13
        AML, DEK::NUP214      1.000     0.833     0.909         6
             AML, ETV6-r      1.000     1.000     1.000         2
           AML, FUS::ERG      1.000     1.000     1.000         1
            AML, KAT6A-r      1.000     1.000     1.000         2
            AML, KMT2A-r      0.918     0.933     0.926        60
            AML, MECOM-r      1.000     1.000     1.000         1
            AML, NPM1alt      0.778     0.955     0.857        22
            AML, NUP98-r      1.000     0.750     0.857        16
        AML,

In [79]:

# ----------------------
# Extract top genes from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Gene": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

gt = GeneThesaurus()
translated = gt.translate_genes(gene_importances["Gene"].tolist(),
                               source="ensembl_id", target="symbol")
gene_importances["Gene"] = gene_importances["Gene"].map(translated)

print("\nTop 20 genes driving classification (final model):")
print(gene_importances.head(20))


Top 20 genes driving classification (final model):
         Gene  Importance
3624    APELA    0.004448
1770   MEGF10    0.004311
1715    MEIS1    0.003837
3081    NUTM1    0.003296
2072    AGAP1    0.003042
3265    GREB1    0.002648
392   PHACTR3    0.002626
2116  IGF2BP1    0.002616
372     MECOM    0.002601
138     HDAC9    0.002520
1193    KRT17    0.002429
517     SALL4    0.002405
1143    LAMP5    0.002384
3638   HOXA10    0.002197
3895  EXOC3L2    0.002073
3098   ANKS1B    0.002065
3549    FADS3    0.002057
2478     EVPL    0.002002
2723    PTPRM    0.001979
690    PTPRZ1    0.001978


In [80]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 5 out of 3924

Bottom 20 genes by importance (final model):
              Gene    Importance
3746           NaN  2.639104e-06
2568         SYAP1  2.603518e-06
3727        TIMM23  2.508661e-06
1047          POLK  2.371750e-06
2179      FAM171A2  2.365607e-06
3166        OR10H1  2.274061e-06
1091        MOGAT1  1.890136e-06
3514           LAT  1.794728e-06
3879          MRM1  1.416080e-06
3196          ERC2  9.027145e-07
3600      KRTAP5-8  8.665183e-07
3666           NaN  6.841172e-07
3857        ZNF670  6.366760e-07
3789        NBPF26  1.784589e-07
3716        NBPF11  1.262941e-20
3008         OR4Q3  0.000000e+00
3709          CCNQ  0.000000e+00
3791           NaN  0.000000e+00
3775        NBPF10  0.000000e+00
3863  SENP3-EIF4A1  0.000000e+00
